# AA-510 Agentic AI Systems (Group 4)

Team Member: Jeffery Smith、Jinyuan He、Mostafa Zamaniturk

**Part A: Agent Deliverable - Technical Artifacts**
- A notebook that contains your data pipeline (DE is the owner)
- A notebook that shows your agent definition (AIE is the owner)
- Review the rubric to determine agent requirements.
- An example of five traces, including the evaluation of the agent (AIE is the owner)
- Traces should be presented via an established provider. Databricks will leverage MLflow by default; alternatives are Arize Phoenix, LangSmith, and Braintrust, among others.
- There should be at least 2 different LLMs used within the agent on the same trace to compare performance (this counts as one trace per the five above).
- Evaluation can be an LLM judge (strongly encouraged) or manual.
- Provide commentary in a notebook cell that talks about the agent performance, and what the evaluation showed, including a comparison between the agent using different LLMs.
- Show 2 examples of the agent gracefully rejecting an irrelevant user input.

# 1. Project Overview

### 1.1 Business problem
### 1.2 Agent goal
### 1.3 Team roles: PM / DE / AIE

# 2. Data Pipeline

### 2.1 Load dataset

In [0]:
%pip install -U -qqqq kaggle backoff databricks-openai uv databricks-agents "mlflow>=3.9" databricks-mcp nest_asyncio "gepa>=0.1.0" "databricks-vectorsearch<0.74"
%pip install -U  sentence-transformers 
dbutils.library.restartPython()

In [0]:
%restart_python

In [0]:
# Download dataset from Kaggle
import os
import pandas as pd

# Create directory if it doesn't exist
os.makedirs('/tmp/ai_reviews', exist_ok=True)

# Download to local tmp directory
! kaggle datasets download -d jahnavikachhia23/the-generative-ai-ecosystem-50k-user-reviews-2026 --unzip -p /tmp/ai_reviews

# Load the CSV into pandas first
pandas_df = pd.read_csv('/tmp/ai_reviews/The_ Generative_AI_Ecosystem_50k_User_Reviews_2026.csv')

# Convert to Spark DataFrame
df = spark.createDataFrame(pandas_df)

display(df)

In [0]:
# Create new catalog "group4"
spark.sql("CREATE CATALOG IF NOT EXISTS group4")

# Save the dataset as a table in the new catalog (using full 3-part name)
df.write.format("delta").mode("overwrite").saveAsTable("group4.default.ai_reviews")

### 2.2 Exploratory Data Analysis

In [0]:
# Exploratory Data Analysis (EDA) on ai_reviews dataset

# Show schema
df.printSchema()

# Show summary statistics for numeric columns
display(df.describe())

# Show first 10 rows
display(df.limit(10))

# Count total rows
row_count = df.count()

# Show distribution of review ratings (if column exists)
if 'rating' in df.columns:
    display(df.groupBy('rating').count().orderBy('rating'))

# Show top 10 most frequent products (if column exists)
if 'product_name' in df.columns:
    display(df.groupBy('product_name').count().orderBy('count', ascending=False).limit(10))

# Show null counts per column
from pyspark.sql.functions import col, sum as spark_sum
null_counts = df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
display(null_counts)

In [0]:
# Aggregate Table - Reviews by Product Sentiment Categories
from pyspark.sql.functions import when, col

# Load the base table
df = spark.table("group4.default.ai_reviews")

# Create sentiment categories based on Sentiment_Polarity
# Positive: > 0.3, Neutral: -0.3 to 0.3, Negative: < -0.3
df_with_sentiment = df.withColumn(
    "sentiment_category",
    when(col("Sentiment_Polarity") > 0.3, "Positive")
    .when(col("Sentiment_Polarity") < -0.3, "Negative")
    .otherwise("Neutral")
)

# Show distribution of sentiment categories
print("Sentiment Category Distribution:")
display(df_with_sentiment.groupBy("sentiment_category").count().orderBy("count", ascending=False))

In [0]:
# Feature Engineering: Temporal Features
from pyspark.sql.functions import to_timestamp, year, month, dayofmonth, dayofweek, hour, quarter, weekofyear

# Parse Review_Date and extract temporal features
df_temporal = df_with_sentiment.withColumn(
    "review_timestamp", to_timestamp(col("Review_Date"), "yyyy-MM-dd HH:mm:ss")
).withColumn(
    "review_year", year(col("review_timestamp"))
).withColumn(
    "review_month", month(col("review_timestamp"))
).withColumn(
    "review_day", dayofmonth(col("review_timestamp"))
).withColumn(
    "review_dayofweek", dayofweek(col("review_timestamp"))  # 1=Sunday, 7=Saturday
).withColumn(
    "review_hour", hour(col("review_timestamp"))
).withColumn(
    "review_quarter", quarter(col("review_timestamp"))
).withColumn(
    "review_week", weekofyear(col("review_timestamp"))
)

# Show sample with temporal features
print("Sample data with temporal features:")
display(df_temporal.select(
    "App", "Review_Date", "review_year", "review_month", "review_day", 
    "review_dayofweek", "review_hour", "Star_Rating"
).limit(10))

In [0]:
# Aggregate Table: Reviews by Product (App)
from pyspark.sql.functions import count, avg, sum as spark_sum, min as spark_min, max as spark_max

# Aggregate by product (App)
df_product_agg = df_temporal.groupBy("App").agg(
    count("*").alias("total_reviews"),
    avg("Star_Rating").alias("avg_rating"),
    avg("Sentiment_Polarity").alias("avg_sentiment"),
    avg("Word_Count").alias("avg_word_count"),
    spark_sum("Thumbs_Up_Count").alias("total_thumbs_up"),
    spark_min("review_timestamp").alias("first_review_date"),
    spark_max("review_timestamp").alias("latest_review_date"),
    count(when(col("sentiment_category") == "Positive", 1)).alias("positive_reviews"),
    count(when(col("sentiment_category") == "Negative", 1)).alias("negative_reviews"),
    count(when(col("sentiment_category") == "Neutral", 1)).alias("neutral_reviews")
).orderBy("total_reviews", ascending=False)

print("Reviews aggregated by Product (App):")
display(df_product_agg)

In [0]:
# Aggregate Table: Reviews by Date
from pyspark.sql.functions import date_format

# Create date column (without time)
df_with_date = df_temporal.withColumn(
    "review_date_only", date_format(col("review_timestamp"), "yyyy-MM-dd")
)

# Aggregate by date
df_date_agg = df_with_date.groupBy("review_date_only").agg(
    count("*").alias("total_reviews"),
    avg("Star_Rating").alias("avg_rating"),
    avg("Sentiment_Polarity").alias("avg_sentiment"),
    spark_sum("Thumbs_Up_Count").alias("total_thumbs_up"),
    count(when(col("sentiment_category") == "Positive", 1)).alias("positive_reviews"),
    count(when(col("sentiment_category") == "Negative", 1)).alias("negative_reviews")
).orderBy("review_date_only", ascending=False)

print("Reviews aggregated by Date:")
display(df_date_agg.limit(20))

# Save aggregate table
df_date_agg.write.format("delta").mode("overwrite").saveAsTable("group4.default.reviews_by_date")
print("\nAggregate table saved to group4.default.reviews_by_date")

In [0]:
# Aggregate by star rating
df_rating_agg = df_temporal.groupBy("Star_Rating").agg(
    count("*").alias("total_reviews"),
    avg("Sentiment_Polarity").alias("avg_sentiment"),
    avg("Word_Count").alias("avg_word_count"),
    avg("Review_Length_Chars").alias("avg_review_length"),
    spark_sum("Thumbs_Up_Count").alias("total_thumbs_up")
).orderBy("Star_Rating")

print("Reviews aggregated by Star Rating:")
display(df_rating_agg)

# Save aggregate table
df_rating_agg.write.format("delta").mode("overwrite").saveAsTable("group4.default.reviews_by_rating")
print("\nAggregate table saved to group4.default.reviews_by_rating")

In [0]:
# Product Category Hierarchy
from pyspark.sql.functions import lit

# Create product category hierarchy based on App names
# Categorize AI apps into types
df_categorized = df_temporal.withColumn(
    "app_category",
    when(col("App").isin(["ChatGPT", "Claude", "Gemini", "Copilot"]), "Conversational AI")
    .when(col("App").isin(["Perplexity", "You.com"]), "Search AI")
    .when(col("App").isin(["Midjourney", "DALL-E", "Stable Diffusion"]), "Image Generation")
    .otherwise("Other AI")
).withColumn(
    "app_provider",
    when(col("App") == "ChatGPT", "OpenAI")
    .when(col("App") == "Claude", "Anthropic")
    .when(col("App") == "Gemini", "Google")
    .when(col("App") == "Copilot", "Microsoft")
    .when(col("App") == "Perplexity", "Perplexity AI")
    .otherwise("Other")
)

# Show distribution by category
print("Distribution by App Category:")
display(df_categorized.groupBy("app_category", "App").count().orderBy("app_category", "count", ascending=False))

In [0]:
# Review theme Hierarchy
from pyspark.sql.functions import lit

# Show distribution by Review_Theme
print("Distribution by App Category:")
display(df_categorized.groupBy("Review_Theme").count().orderBy("Review_Theme", "count", ascending=False))

### 2.3 Create embeddings

In [0]:

from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, ArrayType, FloatType
import pandas as pd
from sentence_transformers import SentenceTransformer

# import model
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
df = spark.table("group4.default.ai_reviews")


In [0]:
# Process embeddings in batches on the driver to avoid worker download issues
# The model is already loaded in cell 19

import pandas as pd
from pyspark.sql.functions import lit, array
from pyspark.sql.types import ArrayType, FloatType

# Get all reviews
df_base = spark.table("group4.default.ai_reviews")
reviews_pdf = df_base.toPandas()

# Generate embeddings in batches
batch_size = 1000
all_embeddings = []

print(f"Processing {len(reviews_pdf)} reviews in batches of {batch_size}...")
for i in range(0, len(reviews_pdf), batch_size):
    batch = reviews_pdf["Review_Text"][i:i+batch_size].tolist()
    batch_embeddings = model.encode(batch, show_progress_bar=False)
    all_embeddings.extend(batch_embeddings.tolist())
    if (i // batch_size + 1) % 10 == 0:
        print(f"Processed {i + len(batch)} reviews...")

print("Embeddings generated. Creating DataFrame...")

# Add embeddings to the pandas DataFrame
reviews_pdf["embedding"] = all_embeddings

# Convert back to Spark DataFrame
df_with_embeddings = spark.createDataFrame(reviews_pdf)

# Write to table
print("Writing to Delta table...")
df_with_embeddings.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("group4.default.ai_reviews_embeddings")

print("✅ Table group4.default.ai_reviews_embeddings created successfully!")

# 3. Tooling Setup

### 3.1 Vector search / RAG tool

In [0]:
# RAG Vector Search Tool
# This function performs semantic similarity search and retrieves relevant reviews
# to provide context for the LLM agent's responses

from pyspark.sql.functions import col, udf, lit, struct
from pyspark.sql.types import DoubleType
import numpy as np
from sentence_transformers import SentenceTransformer
import pandas as pd

# Define cosine similarity UDF at module level (best practice for Spark)
def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors"""
    vec1_arr = np.array(vec1)
    vec2_arr = np.array(vec2)
    dot_product = np.dot(vec1_arr, vec2_arr)
    norm1 = np.linalg.norm(vec1_arr)
    norm2 = np.linalg.norm(vec2_arr)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return float(dot_product / (norm1 * norm2))

cosine_sim_udf = udf(cosine_similarity, DoubleType())

# Model will be loaded on first use to avoid environment sync issues
embedding_model = None

def get_embedding_model():
    """Lazy load the embedding model to avoid UDF serialization issues"""
    global embedding_model
    if embedding_model is None:
        print("Loading SentenceTransformer model...")
        embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
        print("✅ Model loaded successfully")
    return embedding_model

def vector_search(query: str, top_k: int = 10, min_similarity: float = 0.0):
    """
    Perform semantic similarity search on review embeddings.
    
    This is the core RAG retrieval function that:
    1. Converts the query to an embedding vector
    2. Computes cosine similarity with all review embeddings
    3. Returns the top-k most similar reviews
    
    Args:
        query (str): Natural language search query
        top_k (int): Number of top results to return (default: 10)
        min_similarity (float): Minimum similarity threshold 0-1 (default: 0.0)
    
    Returns:
        pd.DataFrame: Top-k most similar reviews with columns:
            - App: Application name
            - Review_Text: Full review text
            - Star_Rating: User rating (1-5)
            - Review_Date: Date of review
            - Sentiment_Polarity: Sentiment score (-1 to 1)
            - Review_Theme: Categorized theme
            - similarity_score: Cosine similarity (0-1)
    """
    # Generate query embedding
    model = get_embedding_model()
    query_embedding = model.encode(query).tolist()
    
    # Load embeddings table
    embeddings_df = spark.table("group4.default.ai_reviews_embeddings")
    
    # Compute similarity scores
    results_df = embeddings_df.withColumn(
        "similarity_score",
        cosine_sim_udf(col("embedding"), lit(query_embedding))
    )
    
    # Filter by minimum similarity and get top_k
    if min_similarity > 0:
        results_df = results_df.filter(col("similarity_score") >= min_similarity)
    
    results_df = results_df.orderBy(col("similarity_score").desc()).limit(top_k)
    
    # Select relevant columns and convert to pandas for agent consumption
    return results_df.select(
        "App",
        "Review_Text",
        "Star_Rating",
        "Review_Date",
        "Sentiment_Polarity",
        "Review_Theme",
        "similarity_score"
    ).toPandas()

def format_context_for_llm(results_df: pd.DataFrame) -> str:
    """
    Format retrieved reviews into a context string for the LLM.
    
    Args:
        results_df (pd.DataFrame): Results from vector_search()
    
    Returns:
        str: Formatted context string with review information
    """
    if len(results_df) == 0:
        return "No relevant reviews found."
    
    context_parts = []
    for idx, row in results_df.iterrows():
        context_parts.append(
            f"Review {idx+1} (Similarity: {row['similarity_score']:.3f}):\n"
            f"App: {row['App']}\n"
            f"Rating: {row['Star_Rating']}/5\n"
            f"Date: {row['Review_Date']}\n"
            f"Theme: {row['Review_Theme']}\n"
            f"Sentiment: {row['Sentiment_Polarity']:.2f}\n"
            f"Review: {row['Review_Text']}\n"
        )
    
    return "\n---\n\n".join(context_parts)

# Test the vector search function
print("\n=== Testing Vector Search Tool ===")
test_query = "What do users complain about app crashes?"
print(f"Query: {test_query}\n")

test_results = vector_search(test_query, top_k=5)
print(f"Found {len(test_results)} similar reviews:\n")
display(test_results[["App", "Star_Rating", "similarity_score", "Review_Theme", "Review_Text"]])

# Show formatted context for LLM
print("\n=== Formatted Context for LLM ===")
context = format_context_for_llm(test_results)
print(context[:1000] + "..." if len(context) > 1000 else context)

### 3.2 Custom functions

In [0]:
%sql
CREATE OR REPLACE FUNCTION group4.default.all_ai_applications_rating()
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Shows all AI applications reveiw count, thumbs up count and average ratings'
RETURN (
  SELECT 
    `App`,
    count(*) as review_count,
    sum(`Thumbs_Up_Count`) as thumbs_up_count,
    avg(`Star_Rating`) as avg_rating
  FROM group4.default.ai_reviews
  GROUP BY 1
  ORDER BY 3 desc
)

In [0]:
%sql
select * from group4.default.all_ai_applications_rating()

In [0]:
%sql
CREATE OR REPLACE FUNCTION group4.default.fetch_review_theme_category()
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Fetch review theme category'
RETURN (
  SELECT 
    `Review_Theme`,
    count(*) as review_count
  FROM group4.default.ai_reviews
  GROUP BY 1
  ORDER BY 2 desc
)

In [0]:
%sql
select * from group4.default.fetch_review_theme_category()

In [0]:
%sql
CREATE OR REPLACE FUNCTION group4.default.fetch_review_contents(
    theme_param STRING COMMENT 'theme category, e.g. Bugs/Performance',
    keyword_param STRING DEFAULT NULL COMMENT 'optional keyword to search in review text'
)
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Fetch review contents by theme, with optional review text keyword'
RETURN (
    SELECT
        `App`,
        `Star_Rating`,
        `Review_Text`,
        `Word_Count`,
        `Review_Length_Chars`,
        `Thumbs_Up_Count`,
        `App_Version`,
        `Sentiment_Polarity`,
        `Review_Theme`
    FROM group4.default.ai_reviews
    WHERE `Review_Theme` = theme_param
      AND (
          keyword_param IS NULL
          OR lower(`Review_Text`) LIKE concat('%', lower(keyword_param), '%')
      )
);

In [0]:
%sql
select * 
from group4.default.fetch_review_contents('Bugs/Performance', 'crash') limit 10

### 3.3 MCP tools if used

In [0]:
from typing import Optional, List, Dict, Any, Callable
import os

# MCP-Style External Tools Manager
# Note: Native MCP connectors (Google Drive, Slack, GitHub, etc.) are only available
# in the Databricks Assistant UI and cannot be accessed programmatically from notebooks.
# This class provides a pattern for adding external API tools to your custom agent.

class ExternalToolsManager:
    """
    Manages external tools integration for the custom agent.
    
    While Databricks MCP connectors are not accessible from custom code,
    you can add external services by:
    1. Direct API calls (shown below)
    2. Unity Catalog external functions
    3. Custom Python functions that wrap external APIs
    """
    
    def __init__(self):
        self.external_tools: List[Dict[str, Any]] = []
    
    def register_external_tool(self, 
                               tool_name: str,
                               description: str,
                               parameters: Dict[str, Any],
                               exec_function: Callable) -> Dict[str, Any]:
        """
        Register an external tool with OpenAI function-calling spec.
        
        Args:
            tool_name: Unique name for the tool
            description: What the tool does
            parameters: OpenAI function parameters schema
            exec_function: Callable that executes the tool
        
        Returns:
            Tool specification dict
        """
        tool_spec = {
            "type": "function",
            "function": {
                "name": tool_name,
                "description": description,
                "parameters": parameters
            }
        }
        
        self.external_tools.append({
            "spec": tool_spec,
            "exec_fn": exec_function
        })
        
        return tool_spec
    
    def get_tool_infos_for_agent(self) -> List[Dict[str, Any]]:
        """
        Get tool specifications in ToolInfo format for the agent.
        
        Returns:
            List of tool info dicts compatible with react_agent.py
        """
        return self.external_tools


# ============================================================================
# Example: Web Search Tool (you can add API key and implement actual search)
# ============================================================================

def web_search_tool(query: str, num_results: int = 5) -> str:
    """
    Example web search tool.
    
    To implement:
    1. Get API key for search service (e.g., Serper, Brave Search, Google Custom Search)
    2. Make API call
    3. Return formatted results
    
    Args:
        query: Search query
        num_results: Number of results to return
    
    Returns:
        Formatted search results
    """
    # Placeholder implementation
    # TODO: Replace with actual API call
    # Example:
    # import requests
    # api_key = dbutils.secrets.get(scope="your-scope", key="search-api-key")
    # response = requests.post(
    #     "https://api.example.com/search",
    #     json={"q": query, "num": num_results},
    #     headers={"Authorization": f"Bearer {api_key}"}
    # )
    # return response.json()
    
    return f"Web search for '{query}' is not yet configured. Add your search API integration here."


# ============================================================================
# Example: Document Retrieval Tool (using Databricks UC Volume)
# ============================================================================

def document_search_tool(query: str, top_k: int = 3) -> str:
    """
    Example document search tool.
    
    Could search through documents stored in UC Volumes or external storage.
    
    Args:
        query: Search query
        top_k: Number of documents to return
    
    Returns:
        Relevant document excerpts
    """
    # Placeholder implementation
    # TODO: Implement document search logic
    # You could:
    # 1. Use vector search on documents stored in UC tables
    # 2. Query a document index
    # 3. Search files in UC Volumes
    
    return f"Document search for '{query}' is not yet configured. Add your document retrieval logic here."


# ============================================================================
# Initialize and Register Tools
# ============================================================================

external_tools_manager = ExternalToolsManager()

# Register web search tool
web_search_spec = external_tools_manager.register_external_tool(
    tool_name="web_search",
    description="Search the web for current information, news, or facts not in the review database",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query"
            },
            "num_results": {
                "type": "integer",
                "description": "Number of results to return (default: 5)",
                "default": 5
            }
        },
        "required": ["query"]
    },
    exec_function=web_search_tool
)

# Register document search tool
doc_search_spec = external_tools_manager.register_external_tool(
    tool_name="document_search",
    description="Search internal documents and knowledge base for relevant information",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query"
            },
            "top_k": {
                "type": "integer",
                "description": "Number of documents to return (default: 3)",
                "default": 3
            }
        },
        "required": ["query"]
    },
    exec_function=document_search_tool
)

print("✅ External tools manager initialized")
print(f"📦 Registered {len(external_tools_manager.external_tools)} external tools:")
for tool in external_tools_manager.external_tools:
    print(f"   - {tool['spec']['function']['name']}: {tool['spec']['function']['description']}")

print("\n💡 To add these tools to your agent:")
print("   1. Import ToolInfo from react_agent.py")
print("   2. For each external tool, create a ToolInfo object")
print("   3. Add to TOOL_INFOS list before agent initialization")
print("\n⚠️  Note: Web search and document search are placeholder implementations.")
print("   Add your actual API keys and logic to make them functional.")

In [0]:
# Bridge: Convert External Tools to Agent-Compatible Format
# This cell creates ToolInfo-compatible objects that can be imported by the agent

from typing import List, Dict, Any

class ExternalToolInfo:
    """
    Simplified ToolInfo structure for external tools.
    Will be converted to actual ToolInfo in react_agent.py
    """
    def __init__(self, name: str, spec: dict, exec_fn):
        self.name = name
        self.spec = spec
        self.exec_fn = exec_fn

# Convert registered external tools to ToolInfo format
EXTERNAL_TOOL_INFOS: List[ExternalToolInfo] = []

for tool_data in external_tools_manager.external_tools:
    tool_info = ExternalToolInfo(
        name=tool_data["spec"]["function"]["name"],
        spec=tool_data["spec"],
        exec_fn=tool_data["exec_fn"]
    )
    EXTERNAL_TOOL_INFOS.append(tool_info)

print(f"\n✅ Converted {len(EXTERNAL_TOOL_INFOS)} external tools to agent-compatible format")
print("\n📋 External tools ready for agent:")
for tool_info in EXTERNAL_TOOL_INFOS:
    print(f"   - {tool_info.name}")

print("\n💡 These tools will be automatically added to the agent in the next cell.")

# 4. Agent Definition

### 4.1 Tools available to the agent

### 4.2 Agent workflow / graph

In [0]:
%%writefile react_agent.py
import copy
import json
import warnings
from typing import Any, Callable, Generator, Optional
from uuid import uuid4

import backoff
import mlflow
import numpy as np
import openai
import pandas as pd
from databricks.sdk import WorkspaceClient
from mlflow.entities import SpanType
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)
from openai import OpenAI
from pydantic import BaseModel
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, udf
from pyspark.sql.types import DoubleType
from sentence_transformers import SentenceTransformer
from databricks_openai import UCFunctionToolkit
from unitycatalog.ai.core.base import get_uc_function_client


############################################
# LLM endpoint — replaced by notebook cell
############################################
LLM_ENDPOINT_NAME = "__LLM_ENDPOINT__"

SYSTEM_PROMPT = """You are an AI Review Analysis Assistant specializing in generative AI application reviews.

Your role is to help users understand user feedback and sentiment about AI applications like
ChatGPT, Claude, Gemini, Midjourney, and other generative AI tools.

You have access to one tool:
- group4__default__all_ai_applications_rating: list all AI applications and their average rating.
- group4__default__fetch_review_theme_category: fetch all review theme categories.
- group4__default__fetch_review_contents: retrieve review content for a specific category, with an optional keyword filter, while limiting the number of records returned per request.


For every user question, follow this process:
1. Think about what information is needed.
2. Choose the correct tool.
3. Review the tool result.
4. If more information is needed, call another tool.
5. Then provide the final answer.

Always ground the final answer in tool results."""


############################################
# Embedding model — lazy loaded once
############################################
_embedding_model: Optional[SentenceTransformer] = None

def _get_embedding_model() -> SentenceTransformer:
    global _embedding_model
    if _embedding_model is None:
        _embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
    return _embedding_model

############################################
# Cosine similarity Spark UDF
############################################
def _cosine_similarity(vec1, vec2) -> float:
    a = np.array(vec1)
    b = np.array(vec2)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom != 0 else 0.0

_cosine_sim_udf = udf(_cosine_similarity, DoubleType())

############################################
# Core retrieval logic
############################################
def _vector_search_exec(
    query: str,
    top_k: int = 10,
    min_similarity: float = 0.0,
) -> str:
    """
    Encode query, compute cosine similarity against ai_reviews_embeddings,
    return formatted context string ready for the LLM.
    """
    spark = SparkSession.builder.getOrCreate()
    model = _get_embedding_model()
    query_vec = model.encode(query).tolist()

    df = spark.table("group4.default.ai_reviews_embeddings")
    df = df.withColumn(
        "similarity_score",
        _cosine_sim_udf(col("embedding"), lit(query_vec)),
    )

    if min_similarity > 0:
        df = df.filter(col("similarity_score") >= min_similarity)

    results: pd.DataFrame = (
        df.orderBy(col("similarity_score").desc())
        .limit(int(top_k))
        .select(
            "App",
            "Review_Text",
            "Star_Rating",
            "Review_Date",
            "Sentiment_Polarity",
            "Review_Theme",
            "similarity_score",
        )
        .toPandas()
    )

    if results.empty:
        return "No relevant reviews found."

    parts = []
    for i, row in results.iterrows():
        parts.append(
            f"Review {i + 1} (Similarity: {row['similarity_score']:.3f}):\n"
            f"App: {row['App']}\n"
            f"Rating: {row['Star_Rating']}/5\n"
            f"Date: {row['Review_Date']}\n"
            f"Theme: {row['Review_Theme']}\n"
            f"Sentiment: {row['Sentiment_Polarity']:.2f}\n"
            f"Review: {row['Review_Text']}\n"
        )
    return "\n---\n\n".join(parts)

############################################
# Tool specs (OpenAI function-calling format)
############################################
_VECTOR_SEARCH_SPEC = {
    "type": "function",
    "function": {
        "name": "vector_search",
        "description": (
            "Search reviews for qualitative topics: complaints, praise, specific user experiences. "
            "Do NOT use for aggregate statistics, rankings, or average ratings — "
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Natural language search query describing what reviews to find",
                },
                "top_k": {
                    "type": "integer",
                    "description": "Number of top results to return (default: 10)",
                    "default": 10,
                },
                "min_similarity": {
                    "type": "number",
                    "description": "Minimum similarity threshold 0-1 (default: 0.0)",
                    "default": 0.0,
                },
            },
            "required": ["query"],
        },
    },
}

############################################
# ToolInfo + agent boilerplate (unchanged)
############################################
class ToolInfo(BaseModel):
    name: str
    spec: dict
    exec_fn: Callable

    class Config:
        arbitrary_types_allowed = True

def create_tool_info(tool_spec, exec_fn_param: Optional[Callable] = None):
    tool_spec["function"].pop("strict", None)
    tool_name = tool_spec["function"]["name"]
    udf_name = tool_name.replace("__", ".")

    # Define a wrapper that accepts kwargs for the UC tool call,
    # then passes them to the UC tool execution client
    def exec_fn(**kwargs):
        function_result = uc_function_client.execute_function(udf_name, kwargs)
        if function_result.error is not None:
            return function_result.error
        else:
            return function_result.value
    return ToolInfo(name=tool_name, spec=tool_spec, exec_fn=exec_fn_param or exec_fn)


def _sanitize_tool_spec(spec: dict) -> dict:
    """Remove JSON schema keywords that some model endpoints reject."""
    spec = copy.deepcopy(spec)
    params = spec.get("function", {}).get("parameters") or {}
    if not isinstance(params, dict) or "properties" not in params:
        return spec
    for prop in params.get("properties", {}).values():
        if not isinstance(prop, dict):
            continue
        t = prop.get("type")
        if t == "string":
            for key in ("minLength", "maxLength", "pattern", "format"):
                prop.pop(key, None)
        elif t in ("integer", "number"):
            for key in ("minimum", "maximum", "exclusiveMinimum", "exclusiveMaximum", "format"):
                prop.pop(key, None)
        elif t == "array":
            for key in ("minItems", "maxItems", "uniqueItems"):
                prop.pop(key, None)
            items = prop.get("items")
            if isinstance(items, dict):
                for key in ("minLength", "maxLength", "pattern", "format",
                            "minimum", "maximum", "exclusiveMinimum", "exclusiveMaximum"):
                    items.pop(key, None)
    return spec


# ---- Register tools here ----
TOOL_INFOS: list[ToolInfo] = [
    # disable vetor search, it is so slow
    # ToolInfo(
    #     name="vector_search",
    #     spec=_VECTOR_SEARCH_SPEC,
    #     exec_fn=_vector_search_exec,
    # ),
]

# Add Unity Catalog functions
UC_TOOL_NAMES = ["group4.default.all_ai_applications_rating", 
                 "group4.default.fetch_review_theme_category", 
                 "group4.default.fetch_review_contents"
                 ]

uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
uc_function_client = get_uc_function_client()
for tool_spec in uc_toolkit.tools:
    TOOL_INFOS.append(create_tool_info(tool_spec))

############################################
# ToolCallingAgent (structure unchanged)
############################################
class ToolCallingAgent(ResponsesAgent):
    def __init__(self, llm_endpoint: str, tools: list[ToolInfo]):
        self.llm_endpoint = llm_endpoint
        self.workspace_client = WorkspaceClient()
        self.model_serving_client: OpenAI = (
            self.workspace_client.serving_endpoints.get_open_ai_client()
        )
        self._tools_dict = {tool.name: tool for tool in tools}

    def get_tool_specs(self) -> list[dict]:
        return [_sanitize_tool_spec(t.spec) for t in self._tools_dict.values()]

    @mlflow.trace(span_type=SpanType.TOOL)
    def execute_tool(self, tool_name: str, args: dict) -> Any:
        sane_args = {k: v for k, v in (args or {}).items() if k and isinstance(k, str)}
        name = tool_name.strip().strip('"').strip("'")
        if "<" in name:
            name = name.split("<")[0].strip()
        if name in self._tools_dict:
            return self._tools_dict[name].exec_fn(**sane_args)
        candidates = [k for k in self._tools_dict if name.startswith(k)]
        if candidates:
            return self._tools_dict[max(candidates, key=len)].exec_fn(**sane_args)
        raise KeyError(f"Unknown tool: {tool_name!r}. Known tools: {list(self._tools_dict.keys())}")

    def call_llm(self, messages: list[dict[str, Any]]) -> Generator[dict[str, Any], None, None]:
        for chunk in self.model_serving_client.chat.completions.create(
            model=self.llm_endpoint,
            messages=to_chat_completions_input(messages),
            tools=self.get_tool_specs(),
            stream=True,
        ):
            chunk_dict = chunk.to_dict()
            if chunk_dict.get("choices"):
                yield chunk_dict

    def handle_tool_call(
        self,
        tool_call: dict[str, Any],
        messages: list[dict[str, Any]],
    ) -> ResponsesAgentStreamEvent:
        try:
            args = json.loads(tool_call.get("arguments"))
        except Exception:
            args = {}
        result = str(self.execute_tool(tool_name=tool_call["name"], args=args))
        tool_call_output = self.create_function_call_output_item(tool_call["call_id"], result)
        messages.append(tool_call_output)
        return ResponsesAgentStreamEvent(type="response.output_item.done", item=tool_call_output)

    def call_and_run_tools(
        self,
        messages: list[dict[str, Any]],
        max_iter: int = 20,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        for _ in range(max_iter):
            last_msg = messages[-1]
            if last_msg.get("role") == "assistant":
                return
            elif last_msg.get("type") == "function_call":
                yield self.handle_tool_call(last_msg, messages)
            else:
                yield from output_to_responses_items_stream(
                    chunks=self.call_llm(messages), aggregator=messages
                )
        yield ResponsesAgentStreamEvent(
            type="response.output_item.done",
            item=self.create_text_output_item("Max iterations reached. Stopping.", str(uuid4())),
        )

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            session_id = None
            if request.custom_inputs and "session_id" in request.custom_inputs:
                session_id = request.custom_inputs.get("session_id")
            elif request.context and request.context.conversation_id:
                session_id = request.context.conversation_id
            if session_id:
                mlflow.update_current_trace(metadata={"mlflow.trace.session": session_id})
            outputs = [
                event.item
                for event in self.predict_stream(request)
                if event.type == "response.output_item.done"
            ]
            return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        session_id = None
        if request.custom_inputs and "session_id" in request.custom_inputs:
            session_id = request.custom_inputs.get("session_id")
        elif request.context and request.context.conversation_id:
            session_id = request.context.conversation_id
        if session_id:
            mlflow.update_current_trace(metadata={"mlflow.trace.session": session_id})
        messages = to_chat_completions_input([i.model_dump() for i in request.input])
        if SYSTEM_PROMPT:
            messages.insert(0, {"role": "system", "content": SYSTEM_PROMPT})
        yield from self.call_and_run_tools(messages=messages)


mlflow.openai.autolog()

In [0]:
# Available foundation model endpoints (adjust for your workspace)
AVAILABLE_LLMS = [
    "databricks-gpt-oss-120b",
    "databricks-gpt-oss-20b",
    "databricks-llama-4-maverick",
]
# Set the LLM: by index (e.g. 0, 1, 2) or by name
CHOSEN_LLM_ENDPOINT = AVAILABLE_LLMS[0]  # change index or set CHOSEN_LLM_ENDPOINT = "your-endpoint-name"
print(f"Using LLM endpoint: {CHOSEN_LLM_ENDPOINT}")

In [0]:
# Bake chosen LLM into agent.py (run "Choose LLM" first)
with open("react_agent.py") as f:
    content = f.read()
with open("react_agent.py", "w") as f:
    f.write(content.replace("__LLM_ENDPOINT__", CHOSEN_LLM_ENDPOINT))
print("react_agent.py updated with LLM_ENDPOINT_NAME =", CHOSEN_LLM_ENDPOINT)

# 5. Agent Test Examples

### 5.1 Normal user queries

In [0]:
import traceback
import nest_asyncio
import sys
import os

# Add the current working directory to Python path so react_agent.py can be imported
current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()
sys.path.insert(0, current_dir)
print(f"Added to Python path: {current_dir}")

import react_agent

nest_asyncio.apply()

# Add external tools (MCP-style) to the agent's tool list
if 'EXTERNAL_TOOL_INFOS' in globals() and EXTERNAL_TOOL_INFOS:
    print(f"\n✅ Adding {len(EXTERNAL_TOOL_INFOS)} external tools to agent...")
    for ext_tool in EXTERNAL_TOOL_INFOS:
        # Convert to react_agent.ToolInfo format
        tool_info = react_agent.ToolInfo(
            name=ext_tool.name,
            spec=ext_tool.spec,
            exec_fn=ext_tool.exec_fn
        )
        react_agent.TOOL_INFOS.append(tool_info)
    print(f"   Total tools available: {len(react_agent.TOOL_INFOS)}")
else:
    print("\nℹ️  No external tools found. Using UC functions only.")
    print(f"   Total tools available: {len(react_agent.TOOL_INFOS)}")

AGENT = react_agent.ToolCallingAgent(llm_endpoint=react_agent.LLM_ENDPOINT_NAME, tools=react_agent.TOOL_INFOS)
import mlflow
mlflow.models.set_model(AGENT)

In [0]:
react_agent.TOOL_INFOS

In [0]:
# Because the free version of Databricks does not support Vector Search indexes, we implemented similarity search using a custom embedding model, BAAI/bge-small-en-v1.5. For each query, the data is reloaded into Spark, so it may take mintues to compelete.

AGENT.predict(
    {"input": [{"role": "user", "content": "What do users complain about app crashes?"}], "custom_inputs": {"session_id": "test-session-123"}},
)

### 5.2 Tool calls

In [0]:
# this will use tool all_ai_applications_rating to reture all AI applications rating
AGENT.predict(
    {"input": [{"role": "user", "content": "List all AI applications and their average rating"}], "custom_inputs": {"session_id": "test-session-123"}},
)

### 5.3 Final answers

# 6. Evaluation & Traces

- At least 5 traces
- LLM judge or manual evaluation
- Success/failure analysis

In [0]:
# Evaluation Test Dataset
# Comprehensive test cases covering different query types and edge cases

import json
from typing import List, Dict, Any

test_dataset = [
    # ===== Factual Queries (Test UC Function Selection) =====
    {
        "query_id": "fact_001",
        "query": "List all AI applications and their average rating",
        "query_type": "factual",
        "expected_tools": ["group4__default__all_ai_applications_rating"],
        "expected_behavior": "Should call all_ai_applications_rating and return complete list",
        "eval_criteria": ["tool_selection", "correctness", "completeness"]
    },
    {
        "query_id": "fact_002",
        "query": "Which AI application has the highest rating?",
        "query_type": "factual",
        "expected_tools": ["group4__default__all_ai_applications_rating"],
        "expected_behavior": "Should identify app with max avg_rating",
        "eval_criteria": ["tool_selection", "correctness", "groundedness"]
    },
    {
        "query_id": "fact_003",
        "query": "What review themes are available in the dataset?",
        "query_type": "factual",
        "expected_tools": ["group4__default__fetch_review_theme_category"],
        "expected_behavior": "Should list all review theme categories",
        "eval_criteria": ["tool_selection", "correctness", "completeness"]
    },
    
    # ===== Theme-Based Queries (Multi-Step Reasoning) =====
    {
        "query_id": "theme_001",
        "query": "What do users complain about regarding bugs and performance?",
        "query_type": "thematic",
        "expected_tools": ["group4__default__fetch_review_contents"],
        "expected_theme": "Bugs/Performance",
        "expected_behavior": "Should fetch Bugs/Performance reviews and synthesize complaints",
        "eval_criteria": ["tool_selection", "groundedness", "relevance", "synthesis"]
    },
    {
        "query_id": "theme_002",
        "query": "Show me reviews about app crashes",
        "query_type": "thematic",
        "expected_tools": ["group4__default__fetch_review_contents"],
        "expected_keyword": "crash",
        "expected_behavior": "Should use keyword filter for 'crash'",
        "eval_criteria": ["tool_selection", "parameter_usage", "groundedness"]
    },
    {
        "query_id": "theme_003",
        "query": "What features do users love about these AI apps?",
        "query_type": "thematic",
        "expected_tools": ["group4__default__fetch_review_contents"],
        "expected_theme": "Features",
        "expected_behavior": "Should fetch positive reviews about features",
        "eval_criteria": ["tool_selection", "groundedness", "sentiment_awareness"]
    },
    
    # ===== Comparison Queries =====
    {
        "query_id": "compare_001",
        "query": "Compare the ratings of ChatGPT and Claude",
        "query_type": "comparison",
        "expected_tools": ["group4__default__all_ai_applications_rating"],
        "expected_behavior": "Should extract and compare ratings for both apps",
        "eval_criteria": ["tool_selection", "correctness", "completeness"]
    },
    
    # ===== Out-of-Scope Queries (Should Reject Gracefully) =====
    {
        "query_id": "reject_001",
        "query": "What's the weather like today?",
        "query_type": "out_of_scope",
        "expected_tools": [],
        "expected_behavior": "Should politely decline and explain scope limitation",
        "eval_criteria": ["rejection_handling", "politeness", "scope_explanation"]
    },
    {
        "query_id": "reject_002",
        "query": "Write me Python code to scrape websites",
        "query_type": "out_of_scope",
        "expected_tools": [],
        "expected_behavior": "Should decline and redirect to review analysis capabilities",
        "eval_criteria": ["rejection_handling", "safety", "redirection"]
    },
]

print(f"✅ Test dataset created with {len(test_dataset)} test cases")
print(f"\nBreakdown by type:")
for qtype in ["factual", "thematic", "comparison", "out_of_scope"]:
    count = len([t for t in test_dataset if t["query_type"] == qtype])
    print(f"  - {qtype}: {count} queries")

print("\n📝 Sample queries:")
for test_case in test_dataset[:3]:
    print(f"  [{test_case['query_id']}] {test_case['query']}")

In [0]:
# LLM-as-Judge Evaluator
# Uses an LLM to evaluate agent responses across multiple dimensions

import openai
import json
import os
from openai import OpenAI
import react_agent

class LLMJudge:
    """LLM-as-judge evaluator for agent responses"""
    
    def __init__(self, llm_endpoint: str):
        # Get Databricks token from context
        try:
            token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
        except:
            # Fallback to environment variable if dbutils not available
            token = os.environ.get("DATABRICKS_TOKEN", "dapi000000000000000000000000000000")
        
        self.client = OpenAI(
            api_key=token,
            base_url=f"https://{spark.conf.get('spark.databricks.workspaceUrl')}/serving-endpoints"
        )
        self.llm_endpoint = llm_endpoint
    
    def evaluate_correctness(self, query: str, tool_results: str, answer: str) -> Dict[str, Any]:
        """Evaluate if the answer is factually correct based on tool results"""
        prompt = f"""You are an expert evaluator. Assess if the agent's answer is factually correct based on the tool results.

User Question: {query}

Tool Results: {tool_results}

Agent's Answer: {answer}

Evaluate the correctness on a scale of 1-5:
5 = Completely correct, all facts match tool results
4 = Mostly correct with minor inaccuracies
3 = Partially correct, some facts wrong
2 = Mostly incorrect
1 = Completely incorrect or hallucinated

Provide your evaluation in JSON format:
{{
  "score": <1-5>,
  "reasoning": "<brief explanation>",
  "errors": ["<list of any factual errors>"]
}}"""
        
        response = self.client.chat.completions.create(
            model=self.llm_endpoint,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=500
        )
        
        try:
            result = json.loads(response.choices[0].message.content)
            return result
        except:
            return {"score": 0, "reasoning": "Failed to parse judge response", "errors": []}
    
    def evaluate_relevance(self, query: str, answer: str) -> Dict[str, Any]:
        """Evaluate if the answer is relevant to the user's question"""
        prompt = f"""You are an expert evaluator. Assess if the answer is relevant to the user's question.

User Question: {query}

Agent's Answer: {answer}

Evaluate relevance on a scale of 1-5:
5 = Perfectly addresses the question
4 = Mostly relevant, minor tangents
3 = Partially relevant
2 = Barely relevant
1 = Completely off-topic

Provide your evaluation in JSON format:
{{
  "score": <1-5>,
  "reasoning": "<brief explanation>"
}}"""
        
        response = self.client.chat.completions.create(
            model=self.llm_endpoint,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=300
        )
        
        try:
            result = json.loads(response.choices[0].message.content)
            return result
        except:
            return {"score": 0, "reasoning": "Failed to parse judge response"}
    
    def evaluate_groundedness(self, tool_results: str, answer: str) -> Dict[str, Any]:
        """Evaluate if the answer is grounded in tool results (no hallucination)"""
        prompt = f"""You are an expert evaluator. Assess if the answer is grounded in the provided tool results or contains hallucinated information.

Tool Results: {tool_results}

Agent's Answer: {answer}

Evaluate groundedness on a scale of 1-5:
5 = All claims are directly supported by tool results
4 = Mostly grounded, minor unsupported claims
3 = Some claims not in tool results
2 = Many unsupported claims
1 = Heavily hallucinated

Provide your evaluation in JSON format:
{{
  "score": <1-5>,
  "reasoning": "<brief explanation>",
  "hallucinations": ["<list of any hallucinated claims>"]
}}"""
        
        response = self.client.chat.completions.create(
            model=self.llm_endpoint,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=400
        )
        
        try:
            result = json.loads(response.choices[0].message.content)
            return result
        except:
            return {"score": 0, "reasoning": "Failed to parse judge response", "hallucinations": []}
    
    def evaluate_rejection(self, query: str, answer: str) -> Dict[str, Any]:
        """Evaluate if out-of-scope queries are rejected gracefully"""
        prompt = f"""You are an expert evaluator. Assess if the agent gracefully rejected this out-of-scope question.

User Question: {query}

Agent's Answer: {answer}

Evaluate the rejection quality on a scale of 1-5:
5 = Politely declined, explained scope, offered alternatives
4 = Politely declined with clear scope explanation
3 = Declined but could be more helpful
2 = Harsh or unhelpful rejection
1 = Attempted to answer despite being out of scope

Provide your evaluation in JSON format:
{{
  "score": <1-5>,
  "reasoning": "<brief explanation>",
  "was_rejected": <true/false>
}}"""
        
        response = self.client.chat.completions.create(
            model=self.llm_endpoint,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=300
        )
        
        try:
            result = json.loads(response.choices[0].message.content)
            return result
        except:
            return {"score": 0, "reasoning": "Failed to parse judge response", "was_rejected": False}

# Initialize judge with the same LLM as the agent
judge = LLMJudge(react_agent.LLM_ENDPOINT_NAME)
print(f"✅ LLM Judge initialized with endpoint: {react_agent.LLM_ENDPOINT_NAME}")

In [0]:
%pip install -U mlflow
dbutils.library.restartPython()

In [0]:
import mlflow
mlflow.openai.autolog()

In [0]:
# Evaluation Runner
# Executes test queries, collects traces, and evaluates responses

import mlflow
import time
import pandas as pd
from typing import List, Dict, Any

class AgentEvaluator:
    """Comprehensive agent evaluation framework"""
    
    def __init__(self, agent, judge: LLMJudge):
        self.agent = agent
        self.judge = judge
        self.results = []
    
    def extract_tool_calls(self, response) -> List[str]:
        """Extract tool names from agent response"""
        tool_calls = []
        if hasattr(response, 'output'):
            for item in response.output:
                if hasattr(item, 'type') and item.type == 'function_call':
                    tool_calls.append(item.name)
        return tool_calls
    
    def extract_tool_results(self, response) -> str:
        """Extract tool results from agent response"""
        results = []
        if hasattr(response, 'output'):
            for item in response.output:
                if hasattr(item, 'type') and item.type == 'function_call_output':
                    results.append(f"Tool output: {str(item.output)[:500]}...")  # Truncate long results
        return "\n".join(results) if results else "No tool results"
    
    def evaluate_tool_selection(self, actual_tools: List[str], expected_tools: List[str]) -> Dict[str, Any]:
        """Evaluate if correct tools were selected"""
        actual_set = set(actual_tools)
        expected_set = set(expected_tools)
        
        if len(expected_tools) == 0:  # Out-of-scope query
            return {
                "precision": 1.0 if len(actual_tools) == 0 else 0.0,
                "recall": 1.0,
                "f1": 1.0 if len(actual_tools) == 0 else 0.0,
                "correct": len(actual_tools) == 0
            }
        
        true_positives = len(actual_set & expected_set)
        precision = true_positives / len(actual_set) if len(actual_set) > 0 else 0.0
        recall = true_positives / len(expected_set) if len(expected_set) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        return {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "correct": actual_set == expected_set,
            "actual_tools": list(actual_set),
            "expected_tools": list(expected_set),
            "missing_tools": list(expected_set - actual_set),
            "extra_tools": list(actual_set - expected_set)
        }
    
    def run_single_evaluation(self, test_case: Dict[str, Any]) -> Dict[str, Any]:
        """Run evaluation on a single test case"""
        query = test_case["query"]
        query_id = test_case["query_id"]
        query_type = test_case["query_type"]
        
        print(f"\n{'='*60}")
        print(f"Evaluating: [{query_id}] {query}")
        print(f"{'='*60}")
        
        # Execute agent
        start_time = time.time()
        try:
            response = self.agent.predict({
                "input": [{"role": "user", "content": query}],
                "custom_inputs": {"session_id": f"eval-{query_id}"}
            })
            latency_ms = (time.time() - start_time) * 1000
            
            # Extract answer text
            if isinstance(response, dict) and "choices" in response:
                answer = response["choices"][0]["message"]["content"]
            else:
                answer = str(response)
            
            # Extract tool calls and results from response
            actual_tools = self.extract_tool_calls(response)
            tool_results = self.extract_tool_results(response)
            
            # Evaluate tool selection
            tool_eval = self.evaluate_tool_selection(
                actual_tools,
                test_case.get("expected_tools", [])
            )
            
            # LLM judge evaluations
            if query_type == "out_of_scope":
                # For rejection queries
                rejection_eval = self.judge.evaluate_rejection(query, answer)
                correctness_eval = {"score": rejection_eval["score"], "reasoning": "Rejection quality"}
                relevance_eval = {"score": rejection_eval["score"], "reasoning": "N/A for rejection"}
                groundedness_eval = {"score": 5, "reasoning": "N/A for rejection"}
            else:
                # For normal queries
                correctness_eval = self.judge.evaluate_correctness(query, tool_results, answer)
                relevance_eval = self.judge.evaluate_relevance(query, answer)
                groundedness_eval = self.judge.evaluate_groundedness(tool_results, answer)
            
            result = {
                "query_id": query_id,
                "query": query,
                "query_type": query_type,
                "answer": answer,
                "answer_length": len(answer),
                "latency_ms": latency_ms,
                "num_tool_calls": len(actual_tools),
                "actual_tools": actual_tools,
                "expected_tools": test_case.get("expected_tools", []),
                "tool_selection_correct": tool_eval["correct"],
                "tool_selection_f1": tool_eval["f1"],
                "correctness_score": correctness_eval["score"],
                "correctness_reasoning": correctness_eval["reasoning"],
                "relevance_score": relevance_eval["score"],
                "relevance_reasoning": relevance_eval["reasoning"],
                "groundedness_score": groundedness_eval["score"],
                "groundedness_reasoning": groundedness_eval["reasoning"],
                "overall_score": (correctness_eval["score"] + relevance_eval["score"] + groundedness_eval["score"]) / 3,
                "trace_id": f"eval-{query_id}",
                "success": True
            }
            
            print(f"\n✅ Success")
            print(f"   Tools: {actual_tools}")
            print(f"   Correctness: {correctness_eval['score']}/5")
            print(f"   Relevance: {relevance_eval['score']}/5")
            print(f"   Groundedness: {groundedness_eval['score']}/5")
            print(f"   Latency: {latency_ms:.0f}ms")
            
            return result
            
        except Exception as e:
            print(f"\n❌ Error: {str(e)}")
            return {
                "query_id": query_id,
                "query": query,
                "query_type": query_type,
                "error": str(e),
                "success": False,
                "latency_ms": (time.time() - start_time) * 1000
            }
    
    def run_evaluation(self, test_cases: List[Dict[str, Any]]) -> pd.DataFrame:
        """Run evaluation on all test cases"""
        print(f"\n{'#'*60}")
        print(f"# Starting evaluation on {len(test_cases)} test cases")
        print(f"{'#'*60}")
        
        self.results = []
        for test_case in test_cases:
            result = self.run_single_evaluation(test_case)
            self.results.append(result)
            time.sleep(1)  # Brief pause between queries
        
        results_df = pd.DataFrame(self.results)
        return results_df

print("✅ AgentEvaluator class defined")

In [0]:
# Run Evaluation on Test Dataset
# This cell executes the evaluation on all test cases and generates metrics

# Initialize evaluator
evaluator = AgentEvaluator(agent=AGENT, judge=judge)

print("\n" + "="*80)
print("STARTING AGENT EVALUATION")
print("="*80)
print(f"\nAgent: {react_agent.LLM_ENDPOINT_NAME}")
print(f"Judge: {judge.llm_endpoint}")
print(f"Test cases: {len(test_dataset)}")
print(f"\nThis will take approximately {len(test_dataset) * 30} seconds...\n")

# Run evaluation
results_df = evaluator.run_evaluation(test_dataset)

# Display results summary
print("\n" + "="*80)
print("EVALUATION COMPLETE")
print("="*80)

# Show successful evaluations
successful_evals = results_df[results_df['success'] == True]
print(f"\n✅ Successful evaluations: {len(successful_evals)} / {len(results_df)}")

if len(successful_evals) > 0:
    print("\n📊 Summary Statistics:")
    print(f"   Average Correctness Score: {successful_evals['correctness_score'].mean():.2f} / 5")
    print(f"   Average Relevance Score: {successful_evals['relevance_score'].mean():.2f} / 5")
    print(f"   Average Groundedness Score: {successful_evals['groundedness_score'].mean():.2f} / 5")
    print(f"   Average Overall Score: {successful_evals['overall_score'].mean():.2f} / 5")
    print(f"   Average Latency: {successful_evals['latency_ms'].mean():.0f} ms")
    print(f"   Average Tool Calls: {successful_evals['num_tool_calls'].mean():.1f}")
    print(f"   Tool Selection Accuracy: {successful_evals['tool_selection_correct'].mean() * 100:.0f}%")

# Show detailed results
print("\n📋 Detailed Results:")
if len(successful_evals) > 0:
    # Show successful evaluations with full metrics
    display(successful_evals[[
        'query_id', 'query_type', 'num_tool_calls', 'tool_selection_correct',
        'correctness_score', 'relevance_score', 'groundedness_score', 
        'overall_score', 'latency_ms'
    ]])
else:
    # Show all results including errors
    print("⚠️  All evaluations failed. Showing errors:")
    display(results_df[['query_id', 'query_type', 'error', 'latency_ms', 'success']])

# Save results
results_df.to_csv('/Workspace' + '/Repos/mzamaniturk@sandiego.edu/aai-510-finalproject-group-4/evaluation_results.csv', index=False)
print("\n💾 Results saved to: evaluation_results.csv")

In [0]:
# Detailed Results Analysis
# Deep dive into evaluation results by query type

import matplotlib.pyplot as plt
import numpy as np

print("\n" + "="*80)
print("DETAILED RESULTS ANALYSIS")
print("="*80)

successful_evals = results_df[results_df['success'] == True]

# 1. Performance by Query Type
print("\n1️⃣ Performance by Query Type:")
print("-" * 60)

for query_type in ['factual', 'thematic', 'comparison', 'out_of_scope']:
    type_results = successful_evals[successful_evals['query_type'] == query_type]
    if len(type_results) > 0:
        print(f"\n{query_type.upper()}:")
        print(f"   Count: {len(type_results)}")
        print(f"   Avg Correctness: {type_results['correctness_score'].mean():.2f} / 5")
        print(f"   Avg Relevance: {type_results['relevance_score'].mean():.2f} / 5")
        print(f"   Avg Groundedness: {type_results['groundedness_score'].mean():.2f} / 5")
        print(f"   Tool Selection Accuracy: {type_results['tool_selection_correct'].mean() * 100:.0f}%")
        print(f"   Avg Latency: {type_results['latency_ms'].mean():.0f} ms")

# 2. Tool Selection Analysis
print("\n2️⃣ Tool Selection Analysis:")
print("-" * 60)

correct_tool_calls = successful_evals[successful_evals['tool_selection_correct'] == True]
incorrect_tool_calls = successful_evals[successful_evals['tool_selection_correct'] == False]

print(f"\n✅ Correct Tool Selection: {len(correct_tool_calls)} cases")
if len(correct_tool_calls) > 0:
    print(f"   Avg Score: {correct_tool_calls['overall_score'].mean():.2f} / 5")
    
print(f"\n❌ Incorrect Tool Selection: {len(incorrect_tool_calls)} cases")
if len(incorrect_tool_calls) > 0:
    print(f"   Avg Score: {incorrect_tool_calls['overall_score'].mean():.2f} / 5")
    print("\n   Cases with incorrect tool selection:")
    for _, row in incorrect_tool_calls.iterrows():
        print(f"   - [{row['query_id']}] {row['query'][:50]}...")
        print(f"     Expected: {row['expected_tools']}")
        print(f"     Got: {row['actual_tools']}")

# 3. Top and Bottom Performing Queries
print("\n3️⃣ Top Performing Queries:")
print("-" * 60)
top_queries = successful_evals.nlargest(3, 'overall_score')
for _, row in top_queries.iterrows():
    print(f"\n[{row['query_id']}] Score: {row['overall_score']:.2f}/5")
    print(f"   Query: {row['query']}")
    print(f"   Correctness: {row['correctness_score']}, Relevance: {row['relevance_score']}, Groundedness: {row['groundedness_score']}")

print("\n4️⃣ Lowest Performing Queries:")
print("-" * 60)
bottom_queries = successful_evals.nsmallest(3, 'overall_score')
for _, row in bottom_queries.iterrows():
    print(f"\n[{row['query_id']}] Score: {row['overall_score']:.2f}/5")
    print(f"   Query: {row['query']}")
    print(f"   Correctness: {row['correctness_score']}, Relevance: {row['relevance_score']}, Groundedness: {row['groundedness_score']}")
    print(f"   Issue: {row.get('correctness_reasoning', 'N/A')}")

# 5. Latency Analysis
print("\n5️⃣ Latency Analysis:")
print("-" * 60)
print(f"   Min Latency: {successful_evals['latency_ms'].min():.0f} ms")
print(f"   Max Latency: {successful_evals['latency_ms'].max():.0f} ms")
print(f"   Median Latency: {successful_evals['latency_ms'].median():.0f} ms")
print(f"   95th Percentile: {successful_evals['latency_ms'].quantile(0.95):.0f} ms")

# 6. Rejection Handling Analysis
print("\n6️⃣ Out-of-Scope Query Handling:")
print("-" * 60)
rejection_cases = successful_evals[successful_evals['query_type'] == 'out_of_scope']
if len(rejection_cases) > 0:
    print(f"\n   Total out-of-scope queries: {len(rejection_cases)}")
    print(f"   Avg rejection score: {rejection_cases['correctness_score'].mean():.2f} / 5")
    print(f"   Tool calls made: {rejection_cases['num_tool_calls'].sum()} (should be 0)")
    print("\n   Examples:")
    for _, row in rejection_cases.iterrows():
        print(f"   - [{row['query_id']}] {row['query']}")
        print(f"     Score: {row['correctness_score']}/5, Tools called: {row['num_tool_calls']}")
else:
    print("   No out-of-scope queries in results")

# 7. LLM Comparison + ROI

- Compare 2 LLMs
- Accuracy / quality difference
- Cost difference
- ROI calculation

In [0]:
# 2-LLM Comparison Evaluation
# Compare two different LLM endpoints for the same agent tasks

import importlib
import sys

print("\n" + "="*80)
print("2-LLM COMPARISON SETUP")
print("="*80)

# Define the two LLMs to compare
LLM_A = "databricks-gpt-oss-120b"  # Larger model
LLM_B = "databricks-gpt-oss-20b"   # Smaller model

print(f"\nLLM A (Baseline): {LLM_A}")
print(f"LLM B (Comparison): {LLM_B}")

# Cost assumptions (adjust based on your pricing)
# Cost per 1M tokens (example values - update with actual pricing)
COST_CONFIG = {
    "databricks-gpt-oss-120b": {
        "input_cost_per_1m": 1.00,   # $ per 1M input tokens
        "output_cost_per_1m": 3.00,  # $ per 1M output tokens
    },
    "databricks-gpt-oss-20b": {
        "input_cost_per_1m": 0.30,   # $ per 1M input tokens
        "output_cost_per_1m": 0.90,  # $ per 1M output tokens
    }
}

def estimate_cost(llm_endpoint: str, avg_input_tokens: int, avg_output_tokens: int) -> float:
    """Estimate cost per query in dollars"""
    config = COST_CONFIG.get(llm_endpoint, COST_CONFIG["databricks-gpt-oss-120b"])
    input_cost = (avg_input_tokens / 1_000_000) * config["input_cost_per_1m"]
    output_cost = (avg_output_tokens / 1_000_000) * config["output_cost_per_1m"]
    return input_cost + output_cost

print("\n📊 Cost Configuration (per 1M tokens):")
for llm, config in COST_CONFIG.items():
    print(f"\n{llm}:")
    print(f"   Input: ${config['input_cost_per_1m']:.2f}")
    print(f"   Output: ${config['output_cost_per_1m']:.2f}")

print("\nℹ️  Note: Update COST_CONFIG with actual pricing from your workspace")

In [0]:
# Run 2-LLM Comparison Evaluation
# Execute the same test cases on both LLMs and compare results

import time

print("\n" + "="*80)
print("RUNNING 2-LLM COMPARISON")
print("="*80)

# Select subset of test cases for comparison (to save time)
comparison_test_cases = [
    test_dataset[0],  # fact_001 - List all AI applications
    test_dataset[1],  # fact_002 - Which app has highest rating
    test_dataset[3],  # theme_001 - What do users complain about bugs
    test_dataset[4],  # theme_002 - Show reviews about crashes
    test_dataset[7],  # reject_001 - Weather question
]

print(f"\nSelected {len(comparison_test_cases)} test cases for comparison")
print("\nTest cases:")
for tc in comparison_test_cases:
    print(f"  - [{tc['query_id']}] {tc['query'][:60]}...")

# Function to run agent with specific LLM
def run_agent_with_llm(llm_endpoint: str, test_cases: list):
    """Create agent with specific LLM and run evaluation"""
    print(f"\n{'='*60}")
    print(f"Testing with: {llm_endpoint}")
    print(f"{'='*60}")
    
    # Create new agent with specified LLM endpoint
    agent_new = react_agent.ToolCallingAgent(
        llm_endpoint=llm_endpoint,
        tools=react_agent.TOOL_INFOS
    )
    
    # Initialize judge with same LLM for consistency
    judge_new = LLMJudge(llm_endpoint)
    
    # Run evaluation
    evaluator_new = AgentEvaluator(agent=agent_new, judge=judge_new)
    results = evaluator_new.run_evaluation(test_cases)
    
    return results

# Run comparison
print("\n🚀 Starting LLM A evaluation...")
results_llm_a = run_agent_with_llm(LLM_A, comparison_test_cases)

print("\n🚀 Starting LLM B evaluation...")
results_llm_b = run_agent_with_llm(LLM_B, comparison_test_cases)

print("\n✅ Both evaluations complete!")

In [0]:
# LLM Comparison Analysis & ROI Calculation
# Compare quality, cost, and ROI between two LLMs

import pandas as pd
import numpy as np

print("\n" + "="*80)
print("LLM COMPARISON ANALYSIS")
print("="*80)

# Filter successful evaluations
llm_a_success = results_llm_a[results_llm_a['success'] == True]
llm_b_success = results_llm_b[results_llm_b['success'] == True]

print(f"\nLLM A ({LLM_A}): {len(llm_a_success)} successful evaluations")
print(f"LLM B ({LLM_B}): {len(llm_b_success)} successful evaluations")

# ===== 1. QUALITY COMPARISON =====
print("\n" + "="*60)
print("1️⃣ QUALITY COMPARISON")
print("="*60)

quality_comparison = pd.DataFrame({
    'Metric': [
        'Correctness Score',
        'Relevance Score',
        'Groundedness Score',
        'Overall Score',
        'Tool Selection Accuracy',
    ],
    f'{LLM_A}': [
        llm_a_success['correctness_score'].mean(),
        llm_a_success['relevance_score'].mean(),
        llm_a_success['groundedness_score'].mean(),
        llm_a_success['overall_score'].mean(),
        llm_a_success['tool_selection_correct'].mean() * 100,
    ],
    f'{LLM_B}': [
        llm_b_success['correctness_score'].mean(),
        llm_b_success['relevance_score'].mean(),
        llm_b_success['groundedness_score'].mean(),
        llm_b_success['overall_score'].mean(),
        llm_b_success['tool_selection_correct'].mean() * 100,
    ]
})

quality_comparison['Difference'] = quality_comparison[f'{LLM_A}'] - quality_comparison[f'{LLM_B}']
quality_comparison['Winner'] = quality_comparison['Difference'].apply(
    lambda x: LLM_A if x > 0 else (LLM_B if x < 0 else 'Tie')
)

print("\n🏆 Quality Metrics (higher is better):")
display(quality_comparison.round(2))

# ===== 2. LATENCY COMPARISON =====
print("\n" + "="*60)
print("2️⃣ LATENCY COMPARISON")
print("="*60)

latency_comparison = pd.DataFrame({
    'Metric': [
        'Average Latency (ms)',
        'Median Latency (ms)',
        'Min Latency (ms)',
        'Max Latency (ms)',
        'P95 Latency (ms)'
    ],
    f'{LLM_A}': [
        llm_a_success['latency_ms'].mean(),
        llm_a_success['latency_ms'].median(),
        llm_a_success['latency_ms'].min(),
        llm_a_success['latency_ms'].max(),
        llm_a_success['latency_ms'].quantile(0.95)
    ],
    f'{LLM_B}': [
        llm_b_success['latency_ms'].mean(),
        llm_b_success['latency_ms'].median(),
        llm_b_success['latency_ms'].min(),
        llm_b_success['latency_ms'].max(),
        llm_b_success['latency_ms'].quantile(0.95)
    ]
})

latency_comparison['Difference (ms)'] = latency_comparison[f'{LLM_A}'] - latency_comparison[f'{LLM_B}']
latency_comparison['Faster'] = latency_comparison['Difference (ms)'].apply(
    lambda x: LLM_B if x > 0 else (LLM_A if x < 0 else 'Tie')
)

print("\n⏱️  Latency Metrics (lower is better):")
display(latency_comparison.round(0))

# ===== 3. COST COMPARISON =====
print("\n" + "="*60)
print("3️⃣ COST COMPARISON")
print("="*60)

# Estimate token usage (approximation: ~3 chars per token for input, ~4 for output)
avg_input_tokens_a = llm_a_success['answer_length'].mean() / 4  # Rough estimate
avg_output_tokens_a = llm_a_success['answer_length'].mean() / 4
avg_input_tokens_b = llm_b_success['answer_length'].mean() / 4
avg_output_tokens_b = llm_b_success['answer_length'].mean() / 4

# For more accurate estimates, use actual context length (query + tool results)
# Assuming average context of ~2000 tokens per query
avg_input_tokens_a = 2000
avg_input_tokens_b = 2000

cost_per_query_a = estimate_cost(LLM_A, avg_input_tokens_a, avg_output_tokens_a)
cost_per_query_b = estimate_cost(LLM_B, avg_input_tokens_b, avg_output_tokens_b)

# Annual cost projection (assuming 1000 queries/day)
queries_per_day = 1000
days_per_year = 365

annual_cost_a = cost_per_query_a * queries_per_day * days_per_year
annual_cost_b = cost_per_query_b * queries_per_day * days_per_year
annual_savings = annual_cost_a - annual_cost_b

cost_comparison_df = pd.DataFrame({
    'Metric': [
        'Cost per Query',
        'Daily Cost (1K queries)',
        'Monthly Cost (30K queries)',
        'Annual Cost (365K queries)'
    ],
    f'{LLM_A}': [
        f"${cost_per_query_a:.6f}",
        f"${cost_per_query_a * queries_per_day:.2f}",
        f"${cost_per_query_a * queries_per_day * 30:.2f}",
        f"${annual_cost_a:.2f}"
    ],
    f'{LLM_B}': [
        f"${cost_per_query_b:.6f}",
        f"${cost_per_query_b * queries_per_day:.2f}",
        f"${cost_per_query_b * queries_per_day * 30:.2f}",
        f"${annual_cost_b:.2f}"
    ],
    'Savings with LLM B': [
        f"${cost_per_query_a - cost_per_query_b:.6f}",
        f"${(cost_per_query_a - cost_per_query_b) * queries_per_day:.2f}",
        f"${(cost_per_query_a - cost_per_query_b) * queries_per_day * 30:.2f}",
        f"${annual_savings:.2f}"
    ]
})

print("\n💰 Cost Analysis:")
display(cost_comparison_df)

# ===== 4. ROI CALCULATION =====
print("\n" + "="*60)
print("4️⃣ ROI ANALYSIS")
print("="*60)

# Quality-adjusted cost (quality score per dollar)
quality_score_a = llm_a_success['overall_score'].mean()
quality_score_b = llm_b_success['overall_score'].mean()

quality_per_dollar_a = quality_score_a / cost_per_query_a if cost_per_query_a > 0 else 0
quality_per_dollar_b = quality_score_b / cost_per_query_b if cost_per_query_b > 0 else 0

roi_a = (quality_score_a / 5) * 100  # Convert to percentage
roi_b = (quality_score_b / 5) * 100

print(f"\n📈 ROI Metrics:")
print(f"\n{LLM_A}:")
print(f"   Quality Score: {quality_score_a:.2f} / 5 ({roi_a:.1f}%)")
print(f"   Cost per Query: ${cost_per_query_a:.6f}")
print(f"   Quality per Dollar: {quality_per_dollar_a:.2f} points/$")
print(f"   Annual Cost (1K queries/day): ${annual_cost_a:.2f}")

print(f"\n{LLM_B}:")
print(f"   Quality Score: {quality_score_b:.2f} / 5 ({roi_b:.1f}%)")
print(f"   Cost per Query: ${cost_per_query_b:.6f}")
print(f"   Quality per Dollar: {quality_per_dollar_b:.2f} points/$")
print(f"   Annual Cost (1K queries/day): ${annual_cost_b:.2f}")

print(f"\n{'='*60}")
print("🎯 RECOMMENDATION:")
print(f"{'='*60}")

quality_diff = quality_score_a - quality_score_b
cost_diff_pct = ((cost_per_query_a - cost_per_query_b) / cost_per_query_a) * 100

if quality_per_dollar_a > quality_per_dollar_b * 1.2:  # 20% better value
    recommendation = f"{LLM_A}"
    reason = f"provides {((quality_per_dollar_a / quality_per_dollar_b - 1) * 100):.0f}% better value (quality per dollar)"
elif quality_per_dollar_b > quality_per_dollar_a * 1.2:
    recommendation = f"{LLM_B}"
    reason = f"provides {((quality_per_dollar_b / quality_per_dollar_a - 1) * 100):.0f}% better value (quality per dollar)"
elif abs(quality_diff) < 0.3:  # Quality is similar
    recommendation = f"{LLM_B}"
    reason = f"provides similar quality ({quality_diff:+.2f} points) at {cost_diff_pct:.0f}% lower cost (${annual_savings:.2f}/year savings)"
else:
    recommendation = f"{LLM_A}"
    reason = f"provides {quality_diff:.2f} points higher quality, worth the additional cost"

print(f"\n⭐ Recommended LLM: {recommendation}")
print(f"   Reason: {reason}")

if annual_savings > 0:
    print(f"\n💵 Potential Annual Savings with {LLM_B}: ${annual_savings:,.2f}")
else:
    print(f"\n💵 Additional Annual Cost with {LLM_A}: ${abs(annual_savings):,.2f}")
    print(f"   Quality improvement: {quality_diff:+.2f} points ({(quality_diff/quality_score_b)*100:+.1f}%)")

# 8. Limitations, Deployment, and Reflection

- What worked well
- What was weak
- Human evaluation role
- Deployment recommendation
- AI tool disclosure